#Лабораторная работа №1

Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):

- Найти велосипед с максимальным временем пробега.
- Найти наибольшее геодезическое расстояние между станциями.
- Найти путь велосипеда с максимальным временем пробега через станции.
- Найти количество велосипедов в системе.
- Найти пользователей потративших на поездки более 3 часов.


In [ ]:
!pip -q install pyspark


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
DATA_DIR = '/content/drive/MyDrive/BD_LR1'

TRIPS_PATH = f'{DATA_DIR}/trips.csv'
STATIONS_PATH = f'{DATA_DIR}/stations.csv'


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('LR1 Bike Analysis RDD')
    .master('local[*]')
    .getOrCreate()
)
sc = spark.sparkContext

print('Spark version:', spark.version)


Spark version: 4.0.2


In [ ]:
import csv
import math
from datetime import datetime

def parse_csv_partition(lines):
    for row in csv.reader(lines):
        yield row

def safe_int(value, default=None):
    value = (value or '').strip()
    if value == '':
        return default
    try:
        return int(float(value))
    except Exception:
        return default

def parse_dt(value):
    value = (value or '').strip()
    if value == '':
        return None
    for fmt in ('%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S'):
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            pass
    return None

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return r * c


In [ ]:
# Чтение trips.csv в RDD
trips_rows = sc.textFile(TRIPS_PATH).mapPartitions(parse_csv_partition)
trips_header = trips_rows.first()
trips_index = {name: i for i, name in enumerate(trips_header)}

def to_trip(row):
    def get(col):
        idx = trips_index[col]
        return row[idx] if idx < len(row) else ''

    start_ts = parse_dt(get('start_date'))
    end_ts = parse_dt(get('end_date'))

    return {
        'id': safe_int(get('id')),
        'duration': safe_int(get('duration'), 0),
        'start_date': get('start_date'),
        'start_ts': start_ts,
        'end_date': get('end_date'),
        'end_ts': end_ts,
        'start_station_name': get('start_station_name'),
        'start_station_id': safe_int(get('start_station_id')),
        'end_station_name': get('end_station_name'),
        'end_station_id': safe_int(get('end_station_id')),
        'bike_id': get('bike_id').strip(),
        'subscription_type': get('subscription_type').strip(),
        'zip_code': get('zip_code').strip(),
    }

trips_rdd = (
    trips_rows
    .filter(lambda row: row != trips_header)
    .map(to_trip)
    .cache()
)

print('Trips count:', trips_rdd.count())


Trips count: 669959


In [ ]:
# Чтение stations.csv в RDD
stations_rows = sc.textFile(STATIONS_PATH).mapPartitions(parse_csv_partition)
stations_header = stations_rows.first()
stations_index = {name: i for i, name in enumerate(stations_header)}

def to_station(row):
    def get(col):
        idx = stations_index[col]
        return row[idx] if idx < len(row) else ''

    return {
        'id': safe_int(get('id')),
        'name': get('name'),
        'lat': float(get('lat')),
        'lon': float(get('long')),
        'dock_count': safe_int(get('dock_count')),
        'city': get('city'),
        'installation_date': get('installation_date'),
    }

stations_rdd = (
    stations_rows
    .filter(lambda row: row != stations_header)
    .map(to_station)
    .cache()
)

print('Stations count:', stations_rdd.count())


Stations count: 70


## 1. Велосипед с максимальным временем пробега

In [ ]:
bike_total_duration = (
    trips_rdd
    .filter(lambda t: t['bike_id'] != '')
    .map(lambda t: (t['bike_id'], t['duration']))
    .reduceByKey(lambda a, b: a + b)
)

max_bike = bike_total_duration.max(key=lambda x: x[1])

max_bike_id = max_bike[0]
max_bike_duration_sec = max_bike[1]
max_bike_duration_hours = max_bike_duration_sec / 3600

print('bike_id =', max_bike_id)
print('total_duration_sec =', max_bike_duration_sec)
print('total_duration_hours =', round(max_bike_duration_hours, 2))


bike_id = 535
total_duration_sec = 18611693
total_duration_hours = 5169.91


## 2. Наибольшее геодезическое расстояние между станциями

In [ ]:
station_pairs = (
    stations_rdd
    .cartesian(stations_rdd)
    .filter(lambda pair: pair[0]['id'] < pair[1]['id'])
)

max_distance_pair = station_pairs.map(
    lambda pair: (
        (
            pair[0]['id'],
            pair[0]['name'],
            pair[1]['id'],
            pair[1]['name']
        ),
        haversine_km(
            pair[0]['lat'], pair[0]['lon'],
            pair[1]['lat'], pair[1]['lon']
        )
    )
).max(key=lambda x: x[1])

pair_info, max_distance_km = max_distance_pair

print('Station 1:', pair_info[0], '-', pair_info[1])
print('Station 2:', pair_info[2], '-', pair_info[3])
print('Max distance (km):', round(max_distance_km, 6))


Station 1: 16 - SJSU - San Salvador at 9th
Station 2: 60 - Embarcadero at Sansome
Max distance (km): 69.920973


## 3. Путь велосипеда с максимальным временем пробега через станции

In [ ]:
bike_path_rdd = (
    trips_rdd
    .filter(lambda t: t['bike_id'] == max_bike_id)
    .sortBy(lambda t: (t['start_ts'] or t['end_ts'] or datetime.max, t['id'] or -1))
    .map(lambda t: (
        t['id'],
        t['start_date'],
        t['start_station_name'],
        t['end_station_name'],
        t['duration']
    ))
)

bike_path = bike_path_rdd.collect()

print('Bike:', max_bike_id)
print('Trips in path:', len(bike_path))
print('First 15 path records:')
for row in bike_path[:15]:
    print(row)


Bike: 535
Trips in path: 1328
First 15 path records:
(4966, '8/29/2013 19:32', 'Post at Kearney', 'San Francisco Caltrain (Townsend at 4th)', 1245)
(5067, '8/29/2013 21:38', 'San Francisco Caltrain (Townsend at 4th)', 'San Francisco Caltrain 2 (330 Townsend)', 423)
(5179, '8/30/2013 8:40', 'San Francisco Caltrain 2 (330 Townsend)', 'Market at Sansome', 842)
(5199, '8/30/2013 9:10', 'Market at Sansome', '2nd at South Park', 498)
(7806, '9/1/2013 12:58', '2nd at Townsend', 'Davis at Jackson', 1671)
(11422, '9/5/2013 11:59', 'San Francisco City Hall', 'Civic Center BART (7th at Market)', 260)
(12245, '9/6/2013 10:55', 'Civic Center BART (7th at Market)', 'Post at Kearney', 1192)
(12485, '9/6/2013 13:58', 'Post at Kearney', 'Embarcadero at Sansome', 1248)
(12558, '9/6/2013 15:07', 'Embarcadero at Sansome', 'Washington at Kearney', 1272)
(13107, '9/6/2013 23:22', 'Washington at Kearney', 'Market at Sansome', 398)
(13423, '9/7/2013 12:08', 'Market at Sansome', 'Market at Sansome', 12476)
(14

## 4. Количество велосипедов в системе

In [ ]:
bike_count = (
    trips_rdd
    .filter(lambda t: t['bike_id'] != '')
    .map(lambda t: t['bike_id'])
    .distinct()
    .count()
)

print('Distinct bikes:', bike_count)


Distinct bikes: 700


## 5. Пользователи, потратившие на поездки более 3 часов

В исходных данных отсутствует отдельный `user_id`, поэтому используем `zip_code`.
Исключаем пустые значения и `nil`.


In [ ]:
users_over_3h_rdd = (
    trips_rdd
    .filter(lambda t: t['zip_code'] != '' and t['zip_code'].lower() != 'nil')
    .map(lambda t: (t['zip_code'], t['duration']))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[1] > 3 * 60 * 60)
    .sortByKey()
    .cache()
)

users_over_3h_count = users_over_3h_rdd.count()
users_over_3h_sample = users_over_3h_rdd.take(20)

print('Users (zip_code) over 3 hours:', users_over_3h_count)
print('First 20:')
for row in users_over_3h_sample:
    print(row)


Users (zip_code) over 3 hours: 3659
First 20:
('0', 314329)
('1', 2471251)
('100', 29323)
('1000', 46574)
('10000', 14614)
('10001', 158530)
('10002', 112885)
('10003', 215175)
('10004', 12464)
('100045', 19125)
('10005', 24881)
('10006', 23966)
('10007', 13516)
('10009', 174866)
('10010', 143724)
('10011', 307222)
('10012', 192523)
('10013', 128624)
('10014', 174262)
('10016', 214172)
